# Date fruit classifier: GPU training sweep (Colab)

1. **Select Kernel → Colab → New Colab Server → L4 or A100 GPU** (T4 works but is slower; **not G4**: TensorFlow cannot run on its Blackwell GPU).
2. **Run All.** Roughly 1–1.5 h on an L4: data download ~5 min, 5 training runs, evaluation, explainability.
3. At the end, download **`/content/results.zip`** (Colab sidebar → Files, or the Colab extension's
   *Download* command) and unzip it into the local `date-fruit-quality-ml/` folder.

Model selection uses the **validation** split only. The test split is scored once at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install pillow-heif
import os, subprocess, time, json, glob, itertools
import tensorflow as tf, keras

# Run a shell command and print its output (only lines containing one of `grep`, if given).
# check=True stops the notebook (Run All) when the command fails.
def sh(cmd, grep=None, check=False):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for line in (out.stdout + out.stderr).splitlines():
        if grep is None or any(g in line for g in grep):
            print(line)
    if check and out.returncode != 0:
        raise RuntimeError(f"command failed (exit {out.returncode}): {cmd}")
    return out.returncode

print("TF", tf.__version__, "| Keras", keras.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU: Runtime → change runtime type → GPU"
# TensorFlow's CUDA kernels don't cover every GPU generation (e.g. Blackwell "G4" fails with
# cuLaunchKernel errors). Run a real op on the GPU now instead of failing mid-sweep.
try:
    with tf.device("/GPU:0"):
        _ = tf.reduce_sum(tf.cast(tf.equal(tf.range(8), 3), tf.float32) + tf.random.normal((256, 256)) @ tf.random.normal((256, 256)))
    print("GPU smoke test OK:", tf.config.experimental.get_device_details(tf.config.list_physical_devices("GPU")[0]))
except Exception as e:
    raise RuntimeError("TensorFlow cannot run on this GPU. Disconnect and pick an L4 or A100 runtime "
                       "(Colab: Remove Server, then New Colab Server -> L4/A100).") from e

## Project source (generated from `src/`, do not edit here)

In [ ]:
%mkdir -p /content/date-fruit-quality-ml/src /content/date-fruit-quality-ml/scripts
%cd /content/date-fruit-quality-ml

In [ ]:
%%writefile src/calibrate.py
"""Confidence calibration and a manual-review threshold for a trained classifier.

1. Temperature scaling (Guo et al., 2017): one scalar T fitted on the validation
   split so that softmax(log p / T) is calibrated. Reported as ECE before/after.
2. Selective prediction: the lowest calibrated-confidence threshold whose
   accuracy on the *validation* predictions it accepts reaches --target. Test
   accuracy on accepted images and the share sent to manual review ("coverage")
   are then measured once on the test split.

T and the threshold are written to outputs/models/<name>_calib.json (name = runs
joined by "+"); grade.py reads them. Several runs are calibrated as an ensemble.

Usage: python src/calibrate.py --run grade_mnv3 grade_effv2b0 [--target 0.98]
"""
import argparse
import json

import numpy as np
import tensorflow as tf
from scipy.optimize import minimize_scalar

import config
from data import load_split
from evaluate import load_run, predict_probs


def scale(probs, T):
    z = np.log(np.clip(probs, 1e-12, 1)) / T
    z = np.exp(z - z.max(1, keepdims=True))
    return z / z.sum(1, keepdims=True)


def nll(probs, y):
    return float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1)).mean())


def ece(probs, y, bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(total)


def coverage_curve(probs, y):
    conf, ok = probs.max(1), probs.argmax(1) == y
    rows = []
    for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.98]:
        m = conf >= t
        rows.append({"threshold": t, "coverage": round(float(m.mean()), 3),
                     "accuracy": round(float(ok[m].mean()), 4) if m.any() else None})
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", nargs="+", required=True)
    ap.add_argument("--target", type=float, default=0.98, help="accuracy wanted on auto-accepted predictions")
    args = ap.parse_args()

    name = config.run_name(args.run)
    out = {"val": [[], None], "test": [[], None]}
    for run in args.run:
        model, meta = load_run(run)
        for split in ("val", "test"):
            ds = load_split(split, meta["img_size"], data_dir=config.DATASETS[meta.get("data", "variety")])
            out[split][1] = np.concatenate([b.numpy() for _, b in ds])
            out[split][0].append(predict_probs(model, ds, tta=True))
        tf.keras.backend.clear_session()
    out = {s: (np.mean(p, axis=0), y) for s, (p, y) in out.items()}

    pv, yv = out["val"]
    T = float(minimize_scalar(lambda t: nll(scale(pv, t), yv), bounds=(0.05, 10), method="bounded").x)
    cal = {s: scale(p, T) for s, (p, _) in out.items()}

    # lowest threshold reaching the target on val (more coverage = less manual work)
    conf_v, ok_v = cal["val"].max(1), cal["val"].argmax(1) == yv
    threshold = None
    for t in np.unique(conf_v):
        m = conf_v >= t
        if ok_v[m].mean() >= args.target:
            threshold = float(t)
            break
    pt, yt = cal["test"], out["test"][1]
    acc_t = pt.argmax(1) == yt
    accepted = pt.max(1) >= threshold if threshold is not None else np.zeros(len(yt), bool)

    report = {
        "run": name,
        "temperature": round(T, 4),
        "val": {"nll_before": round(nll(pv, yv), 4), "nll_after": round(nll(cal["val"], yv), 4),
                "ece_before": round(ece(pv, yv), 4), "ece_after": round(ece(cal["val"], yv), 4)},
        "test": {"accuracy": round(float(acc_t.mean()), 4),
                 "nll_before": round(nll(out["test"][0], yt), 4), "nll_after": round(nll(pt, yt), 4),
                 "ece_before": round(ece(out["test"][0], yt), 4), "ece_after": round(ece(pt, yt), 4)},
        "review": {
            "target_accuracy": args.target,
            "threshold": None if threshold is None else round(threshold, 4),
            "test_coverage": round(float(accepted.mean()), 4),
            "test_accuracy_on_accepted": round(float(acc_t[accepted].mean()), 4) if accepted.any() else None,
            "test_sent_to_review": int((~accepted).sum()),
            "n_test": int(len(yt)),
        },
        "test_coverage_curve": coverage_curve(pt, yt),
    }
    (config.MODEL_DIR / f"{name}_calib.json").write_text(json.dumps(
        {"runs": args.run, "temperature": report["temperature"], "review_threshold": report["review"]["threshold"],
         "target_accuracy": args.target}, indent=2))
    dest = config.OUTPUT_DIR / "eval" / f"{name}_calibration.json"
    dest.write_text(json.dumps(report, indent=2))
    print(json.dumps({k: v for k, v in report.items() if k != "test_coverage_curve"}, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/config.py
"""Central configuration for the date fruit project (Phase 1: variety, Phase 2: quality)."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW_DIR = ROOT / "data" / "raw"  # Zenodo 4639543, flat folder
ALHAMDAN_DIR = ROOT / "data" / "kaggle" / "alhamdan"  # Kaggle "Date Fruit Image Dataset in Controlled Environment"
GRADING_DIR = ROOT / "data" / "kaggle" / "grading"  # variety/size/grade dataset, for Phase 2
PROCESSED_DIR = ROOT / "data" / "processed"
SPLITS_CSV = ROOT / "data" / "splits.csv"
OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
MATURITY_SRC_DIR = ROOT / "data" / "maturity"  # Zenodo 10143465 crops, from scripts/download_maturity.py
QUALITY_DIR = ROOT / "data" / "quality"  # Phase 2 splits, from src/prepare_quality.py

# Image-classification datasets (train.py --data): name -> <dir>/<split>/<class>/*.jpg
DATASETS = {
    "variety": PROCESSED_DIR,
    "grade": QUALITY_DIR / "grade",
    "maturity": QUALITY_DIR / "maturity",
}

# Zenodo filename prefix -> class name used by the doc.
# The Zenodo record actually publishes 478 images across ~10 prefixes, not the
# 27 classes / 2000+ images its description claims. "Sawr" (7 images) and
# "IMG_*" (unlabelled) are never used.
ZENODO_MAP = {
    "NbotAli": "NbotAli",
    "SequeeIRAQ": "SagaiIRAQ",
    "Skari": "Al-skari",
    "skri_magrosh": "Al-skri_magrosh",
    "Muraaya": "Al-muraaya",
    "sequee": "Al-sagai",
    "masyihia": "Al-masyihia",
    "Shagra": "Al-shagra",
}
# Alhamdan folder -> class name. "Rutab" is left out: in the doc Rutab is a
# ripeness stage (Khalal/Rutab/Tamar), not a variety.
ALHAMDAN_MAP = {
    "Ajwa": "Al-ajwa",
    "Medjool": "Al-majdool",
    "Sokari": "Al-skari",
    "Sugaey": "Al-sagai",
    "Nabtat Ali": "NbotAli",
    "Meneifi": "Al-meneifi",
    "Shaishe": "Al-shaishe",
    "Galaxy": "Galaxy",
}

# The 10 varieties trained. First five are on the doc's preferred list; the
# rest are the best-supplied remaining classes.
CLASSES = [
    "Al-skari", "Al-ajwa", "Al-majdool", "Al-sagai", "Al-muraaya",
    "NbotAli", "Al-meneifi", "Al-shaishe", "Galaxy", "Al-skri_magrosh",
]

# Longest side of images stored in data/processed (originals go up to ~9 MB).
STORE_MAX_SIDE = 640
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}

# "mobilenetv3" (MobileNetV3Large) or "efficientnet" (EfficientNetB0;
# EfficientNet-Lite has no official Keras application, B0 is its closest counterpart).
BACKBONE = "mobilenetv3"
HEAD_EPOCHS = 25
FINETUNE_EPOCHS = 30
HEAD_LR = 1e-3
FINETUNE_LR = 3e-5
FINETUNE_LAST_N_LAYERS = 60

# ---------------- Phase 2: quality ----------------
MATURITY_STAGES = ["Immature", "Khalal", "Rutab", "Tamar"]
# Moroccan variety held out entirely for the maturity test split (orchard sessions
# are one per stage, so a random split would leak session lighting into test).
MATURITY_TEST_VARIETY = "kholt"
# Printed ArUco marker (cv2.aruco DICT_4X4_50) placed next to the fruit gives the mm/px scale.
ARUCO_DICT = "DICT_4X4_50"
ARUCO_SIDE_MM = 30.0
# Weight = k * length * width^2 (mm -> g). An ellipsoid of revolution has V = pi/6 * L * W^2;
# with date flesh density ~1.3 g/cm^3 that gives k ~ 6.8e-4. Recalibrate per variety with
# `python src/quality.py --calibrate-weight <csv>` or a bag weight (see grade.py --bag-weight).
WEIGHT_K = 6.8e-4

# Models used by grade.py (chosen on the validation split; see reports/model_report.html).
# A list of several runs = an ensemble (averaged probabilities).
PRODUCTION_RUNS = {
    "variety": ["effv2b0_260"],
    "grade": ["grade_mnv3", "grade_effv2b0"],
    "maturity": ["maturity_effv2b0"],
}


def run_name(runs):
    """File-name key of a model or ensemble, as written by evaluate.py / calibrate.py."""
    return "+".join([runs] if isinstance(runs, str) else runs)

In [ ]:
%%writefile src/data.py
"""tf.data input pipeline with augmentation. Images stay in [0, 255]; every
backbone includes its own rescaling layer."""
from pathlib import Path

import tensorflow as tf

from config import BATCH_SIZE, PROCESSED_DIR, SEED

AUTOTUNE = tf.data.AUTOTUNE


def build_augmenter():
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal_and_vertical"),
            tf.keras.layers.RandomRotation(0.25, fill_mode="reflect"),
            tf.keras.layers.RandomZoom((-0.15, 0.25), fill_mode="reflect"),
            tf.keras.layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
            tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
            tf.keras.layers.RandomContrast(0.15),
        ],
        name="augment",
    )


def load_split(split, img_size, shuffle=False, batch_size=BATCH_SIZE, data_dir=PROCESSED_DIR):
    return tf.keras.utils.image_dataset_from_directory(
        Path(data_dir) / split,
        labels="inferred",
        label_mode="int",
        image_size=(img_size, img_size),
        crop_to_aspect_ratio=True,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=SEED,
    )


def get_datasets(img_size, batch_size=BATCH_SIZE, one_hot=False, data_dir=PROCESSED_DIR):
    train = load_split("train", img_size, shuffle=True, batch_size=batch_size, data_dir=data_dir)
    val = load_split("val", img_size, batch_size=batch_size, data_dir=data_dir)
    test = load_split("test", img_size, batch_size=batch_size, data_dir=data_dir)
    class_names = train.class_names
    assert val.class_names == class_names == test.class_names
    n = len(class_names)
    enc = (lambda y: tf.one_hot(y, n)) if one_hot else (lambda y: y)

    augment = build_augmenter()
    train = train.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), enc(y)),
                      num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    val = val.map(lambda x, y: (x, enc(y))).cache().prefetch(AUTOTUNE)
    test = test.map(lambda x, y: (x, enc(y))).cache().prefetch(AUTOTUNE)
    return train, val, test, class_names

In [ ]:
%%writefile src/evaluate.py
"""Evaluate trained model(s): loss, accuracy, F1, per-class report and confusion matrix.

  --tta          average predictions over 4 flips of each image
  --run a b ...  several runs = ensemble (averaged probabilities)
  --split val    score on validation instead of test (use this for model selection)

Usage: python src/evaluate.py --run mobilenetv3 [--tta] [--split test]
"""
import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score

import config
from data import load_split

FLIPS = [lambda x: x, lambda x: tf.reverse(x, [2]), lambda x: tf.reverse(x, [1]), lambda x: tf.reverse(x, [1, 2])]


def load_run(run):
    model = tf.keras.models.load_model(config.MODEL_DIR / f"{run}_best.keras")
    meta = json.loads((config.MODEL_DIR / f"{run}_meta.json").read_text())
    return model, meta


def predict_probs(model, ds, tta):
    views = FLIPS if tta else FLIPS[:1]
    return np.mean([model.predict(ds.map(lambda x, y, f=f: (f(x), y)), verbose=0) for f in views], axis=0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", nargs="+", default=[config.BACKBONE])
    ap.add_argument("--tta", action="store_true")
    ap.add_argument("--split", default="test", choices=["val", "test"])
    args = ap.parse_args()

    probs, class_names, y_true, files = [], None, None, None
    for run in args.run:
        model, meta = load_run(run)
        ds = load_split(args.split, meta["img_size"], data_dir=config.DATASETS[meta.get("data", "variety")])
        assert class_names is None or meta["class_names"] == class_names
        class_names, files = meta["class_names"], ds.file_paths
        y_true = np.concatenate([y.numpy() for _, y in ds])
        probs.append(predict_probs(model, ds, args.tta))
        tf.keras.backend.clear_session()
    probs = np.mean(probs, axis=0)
    y_pred = probs.argmax(1)
    sources = np.array([Path(p).name.split("__")[0] for p in files])
    name = "+".join(args.run) + ("_tta" if args.tta else "")

    metrics = {
        "name": name,
        "split": args.split,
        "n": int(len(y_true)),
        "loss": float(tf.keras.losses.SparseCategoricalCrossentropy()(y_true, probs)),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "top2_accuracy": float(np.mean([t in p.argsort()[-2:] for t, p in zip(y_true, probs)])),
        # Filename prefix before "__" = photo source (variety data) or variety (grade/maturity data).
        "accuracy_by_source": {
            str(s): float(np.mean((y_pred == y_true)[sources == s])) for s in np.unique(sources)
        },
        "per_class": classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0),
        "misclassified": [
            {"file": Path(f).name, "true": class_names[t], "pred": class_names[p], "conf": round(float(pr[p]), 3)}
            for f, t, p, pr in zip(files, y_true, y_pred, probs) if t != p
        ],
    }
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))
    print(f"[{name} | {args.split}]", {k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})
    print("accuracy by source:", {k: round(v, 4) for k, v in metrics["accuracy_by_source"].items()})

    out = config.OUTPUT_DIR / "eval"
    out.mkdir(parents=True, exist_ok=True)
    (out / f"{name}_{args.split}_metrics.json").write_text(json.dumps(metrics, indent=2))
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    fig, ax = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
    ax.set_title(f"{args.split} confusion matrix ({name})\nacc {metrics['accuracy']:.3f}, macro-F1 {metrics['f1_macro']:.3f}")
    fig.tight_layout()
    fig.savefig(out / f"{name}_{args.split}_confusion_matrix.png", dpi=120)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/explain.py
"""Explainability: Grad-CAM, Grad-CAM++ and Guided Grad-CAM, plus sanity checks.

Guided Grad-CAM = Grad-CAM x a fine-grained input-gradient map. Classic guided
backprop needs every ReLU's gradient overridden, which MobileNetV3's
hard-swish blocks don't allow cleanly, so the fine-grained map here is
SmoothGrad (positive input gradients averaged over noisy copies).

Sanity checks (per image, summarised over the test set):
  * Model-randomisation test (Adebayo et al., 2018): heatmaps from the trained
    model vs the same architecture with (a) a re-initialised head, (b) all
    weights re-initialised. Low Spearman correlation = the map depends on what
    the model learned, i.e. it is not just an edge detector.
  * Deletion test: confidence drop when the top-20% heatmap pixels are blurred
    out, vs blurring 20% random pixels. A faithful map causes a bigger drop.
  * Fruit-focus: share of heatmap energy that falls on fruit pixels
    (GrabCut mask). Low values mean the model looks at the background, which is a
    real risk here because each Zenodo class was shot in its own session.

Usage: python src/explain.py [--run mobilenetv3] [--per-class 2]
"""
import argparse
import json

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from scipy.stats import spearmanr

import config
from features import segment_fruit


def _split(model):
    return model.get_layer("backbone"), model.get_layer("head")


def _grads(model, x, class_idx):
    backbone, head = _split(model)
    with tf.GradientTape() as tape:
        fmap = backbone(x, training=False)
        tape.watch(fmap)
        score = head(fmap, training=False)[:, class_idx]
    return fmap[0].numpy(), tape.gradient(score, fmap)[0].numpy()


def _norm(cam):
    cam = np.maximum(cam, 0)
    return cam / cam.max() if cam.max() > 0 else cam


def gradcam(model, x, class_idx):
    fmap, g = _grads(model, x, class_idx)
    return _norm((fmap * g.mean((0, 1))).sum(-1))


def gradcam_pp(model, x, class_idx):
    fmap, g = _grads(model, x, class_idx)
    g2, g3 = g**2, g**3
    denom = 2 * g2 + fmap.sum((0, 1)) * g3
    alpha = g2 / np.where(denom != 0, denom, 1e-10)
    weights = (alpha * np.maximum(g, 0)).sum((0, 1))
    return _norm((fmap * weights).sum(-1))


def smoothgrad(model, x, class_idx, n=24, sigma=0.12):
    xs = tf.repeat(x, n, 0) + tf.random.normal((n, *x.shape[1:]), stddev=sigma * 255)
    with tf.GradientTape() as tape:
        tape.watch(xs)
        score = model(xs, training=False)[:, class_idx]
    g = tape.gradient(score, xs).numpy().mean(0)
    g = np.maximum(g, 0).sum(-1)
    return g / g.max() if g.max() > 0 else g


def upsample(cam, size):
    return cv2.resize(cam.astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC).clip(0, 1)


def overlay(img, cam, alpha=0.45):
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1]
    return np.uint8((1 - alpha) * img + alpha * heat)


def randomized_copy(model, head_only):
    rnd = tf.keras.models.clone_model(model)  # fresh random initialisation
    if head_only:
        rnd.get_layer("backbone").set_weights(model.get_layer("backbone").get_weights())
    return rnd


def rank_corr(a, b):
    """Spearman correlation; a constant (vanished) map counts as uncorrelated."""
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return 0.0
    return float(spearmanr(a.ravel(), b.ravel())[0])


def deletion_drop(model, x, cam, class_idx, frac=0.2, rng=None):
    img = x[0].numpy()
    blurred = cv2.GaussianBlur(img, (0, 0), 12)
    p0 = float(model(x, training=False)[0, class_idx])
    k = int(frac * cam.size)
    top = np.zeros(cam.size, bool)
    top[np.argsort(cam.ravel())[-k:]] = True
    rand = np.zeros(cam.size, bool)
    rand[rng.choice(cam.size, k, replace=False)] = True
    out = []
    for m in (top, rand):
        m = m.reshape(cam.shape)[..., None]
        xm = np.where(m, blurred, img)[None]
        out.append(p0 - float(model(xm, training=False)[0, class_idx]))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default=config.BACKBONE)
    ap.add_argument("--per-class", type=int, default=2, help="example images per class in the gallery")
    args = ap.parse_args()

    model = tf.keras.models.load_model(config.MODEL_DIR / f"{args.run}_best.keras")
    meta = json.loads((config.MODEL_DIR / f"{args.run}_meta.json").read_text())
    class_names = meta["class_names"]
    rnd_head, rnd_all = randomized_copy(model, True), randomized_copy(model, False)
    out_dir = config.OUTPUT_DIR / "explainability"
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(config.SEED)
    S = meta.get("img_size", config.IMG_SIZE)

    stats, gallery = [], []
    for ci, cname in enumerate(class_names):
        paths = sorted((config.DATASETS[meta.get("data", "variety")] / "test" / cname).glob("*.jpg"))
        for i, p in enumerate(paths):
            img = tf.keras.utils.img_to_array(
                tf.keras.utils.load_img(p, target_size=(S, S), keep_aspect_ratio=True)
            )
            x = tf.constant(img[None])
            probs = model(x, training=False)[0].numpy()
            pred = int(probs.argmax())
            cam = upsample(gradcam(model, x, pred), S)
            campp = upsample(gradcam_pp(model, x, pred), S)
            guided = cam * smoothgrad(model, x, pred)
            guided = guided / guided.max() if guided.max() > 0 else guided

            cam_rh = upsample(gradcam(rnd_head, x, pred), S)
            cam_ra = upsample(gradcam(rnd_all, x, pred), S)
            fruit = segment_fruit(img.astype(np.uint8)) > 0
            drop_cam, drop_rand = deletion_drop(model, x, cam, pred, rng=rng)
            stats.append({
                "file": p.name, "true": cname, "pred": class_names[pred], "conf": float(probs[pred]),
                "spearman_vs_random_head": rank_corr(cam, cam_rh),
                "spearman_vs_random_all": rank_corr(cam, cam_ra),
                "deletion_drop_cam": drop_cam, "deletion_drop_random": drop_rand,
                "cam_energy_on_fruit": float(cam[fruit].sum() / max(cam.sum(), 1e-9)),
                "fruit_area_fraction": float(fruit.mean()),
            })
            if i < args.per_class:
                gallery.append((img.astype(np.uint8), cam, campp, guided, cam_ra, stats[-1]))

    # Gallery: original | Grad-CAM | Grad-CAM++ | Guided Grad-CAM | randomised-model Grad-CAM
    cols = ["input", "Grad-CAM", "Grad-CAM++", "Guided Grad-CAM", "Grad-CAM (random weights)"]
    fig, axes = plt.subplots(len(gallery), 5, figsize=(15, 3.1 * len(gallery)))
    for r, (img, cam, campp, guided, cam_ra, s) in enumerate(gallery):
        panels = [img, overlay(img, cam), overlay(img, campp), guided, overlay(img, cam_ra)]
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(panel, cmap="inferno" if c == 3 else None)
            ax.set_xticks([]), ax.set_yticks([])
            if r == 0:
                ax.set_title(cols[c], fontsize=11)
        ok = "✓" if s["true"] == s["pred"] else "✗"
        axes[r, 0].set_ylabel(f"{s['true']}\n→ {s['pred']} {s['conf']:.2f} {ok}", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_dir / f"{args.run}_gradcam_gallery.png", dpi=90)

    keys = ["spearman_vs_random_head", "spearman_vs_random_all", "deletion_drop_cam",
            "deletion_drop_random", "cam_energy_on_fruit", "fruit_area_fraction"]
    summary = {k: round(float(np.mean([s[k] for s in stats])), 4) for k in keys}
    summary["verdict"] = {
        "depends_on_learned_weights": summary["spearman_vs_random_all"] < 0.5,
        "faithful_deletion": summary["deletion_drop_cam"] > summary["deletion_drop_random"],
        "focuses_on_fruit": summary["cam_energy_on_fruit"] > summary["fruit_area_fraction"],
    }
    (out_dir / f"{args.run}_sanity_checks.json").write_text(
        json.dumps({"summary": summary, "per_image": stats}, indent=2))
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/features.py
"""Hand-crafted colour / shape / texture descriptors of the fruit in an image.

The fruit is segmented with GrabCut (initialised from a centred rectangle), so
the numbers describe fruit pixels only, not the background. Shape values are in
pixels of a 512-px-long image: without a scale reference in the photo they are
relative, not physical, sizes.

Usage:
  python src/features.py <image>            # print features of one image
  python src/features.py --class-profiles   # per-class profile over the train split
"""
import argparse
import json
import sys

import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.cluster import KMeans

WORK_SIDE = 512


def load_rgb(path):
    from PIL import Image, ImageOps
    from pillow_heif import register_heif_opener

    register_heif_opener()
    img = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert("RGB"))
    scale = WORK_SIDE / max(img.shape[:2])
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)


def _plain_background_mask(rgb):
    """Fast path for studio photos: pixels far (in Lab) from the border colour.
    Returns None when the border is not a plain background."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    border = np.concatenate([lab[:6].reshape(-1, 3), lab[-6:].reshape(-1, 3),
                             lab[:, :6].reshape(-1, 3), lab[:, -6:].reshape(-1, 3)])
    if border.std(0).max() > 12:
        return None
    dist = np.linalg.norm(lab - np.median(border, 0), axis=2)
    dist = cv2.GaussianBlur(np.clip(dist * 4, 0, 255).astype(np.uint8), (5, 5), 0)
    _, mask = cv2.threshold(dist, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask


def segment_fruit(rgb):
    """Binary mask (uint8 0/255) of fruit pixels."""
    fast = _plain_background_mask(rgb)
    if fast is not None:
        return _clean(fast)
    h, w = rgb.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    rect = (int(w * 0.04), int(h * 0.04), int(w * 0.92), int(h * 0.92))
    bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
    cv2.grabCut(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), mask, rect, bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
    fruit = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    return _clean(fruit)


def _clean(fruit):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    fruit = cv2.morphologyEx(cv2.morphologyEx(fruit, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
    # Keep blobs that are at least 10% of the largest one (drops specks).
    n, labels, stats, _ = cv2.connectedComponentsWithStats(fruit)
    if n <= 1:
        return fruit
    areas = stats[1:, cv2.CC_STAT_AREA]
    keep = 1 + np.flatnonzero(areas >= 0.1 * areas.max())
    fruit = np.isin(labels, keep).astype(np.uint8) * 255
    # Fill holes (specular highlights can look like background).
    contours, _ = cv2.findContours(fruit, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.drawContours(fruit, contours, -1, 255, cv2.FILLED)


def color_name(h_deg, s, v):
    if v < 0.18 or (v < 0.33 and s < 0.35):
        return "black / very dark brown"
    if s < 0.18:
        return "greyish / dull"
    if h_deg < 12 or h_deg >= 340:
        return "dark red-brown" if v < 0.45 else "reddish brown"
    if h_deg < 25:
        return "dark brown" if v < 0.4 else "amber brown"
    if h_deg < 45:
        return "golden brown" if v < 0.6 else "amber / golden yellow"
    if h_deg < 70:
        return "yellow"
    return "greenish yellow"


def color_features(rgb, mask):
    px = rgb[mask > 0]
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV_FULL)[mask > 0].astype(float)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)[mask > 0].astype(float)
    lab[:, 0] *= 100 / 255
    lab[:, 1:] -= 128
    # circular mean for hue
    ang = hsv[:, 0] / 255 * 2 * np.pi
    hue = (np.degrees(np.arctan2(np.sin(ang).mean(), np.cos(ang).mean())) + 360) % 360
    sat, val = hsv[:, 1].mean() / 255, hsv[:, 2].mean() / 255

    km = KMeans(n_clusters=3, n_init=4, random_state=0).fit(px[:: max(1, len(px) // 5000)].astype(float))
    share = np.bincount(km.labels_, minlength=3) / len(km.labels_)
    order = share.argsort()[::-1]
    dominant = [
        {"hex": "#%02x%02x%02x" % tuple(int(c) for c in km.cluster_centers_[i]), "share": round(float(share[i]), 3)}
        for i in order
    ]
    # Colour uniformity: spread of Lab colour around its mean (mean ΔE76), mapped to 0-1.
    delta_e = np.linalg.norm(lab - lab.mean(0), axis=1).mean()
    return {
        "color_name": color_name(hue, sat, val),
        "mean_rgb": [round(float(c), 1) for c in px.mean(0)],
        "mean_hue_deg": round(float(hue), 1),
        "mean_saturation": round(float(sat), 3),
        "mean_brightness": round(float(val), 3),
        "mean_lab": [round(float(c), 1) for c in lab.mean(0)],
        "dominant_colors": dominant,
        "color_spread_deltaE": round(float(delta_e), 2),
        "color_uniformity": round(float(np.exp(-delta_e / 15)), 3),
    }


def shape_features(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    big = [c for c in contours if cv2.contourArea(c) > 0]
    c = max(big, key=cv2.contourArea)
    area, perim = cv2.contourArea(c), cv2.arcLength(c, True)
    hull_area = cv2.contourArea(cv2.convexHull(c))
    x, y, w, h = cv2.boundingRect(c)
    if len(c) >= 5:
        (_, _), (a1, a2), _ = cv2.fitEllipse(c)
        major, minor = max(a1, a2), min(a1, a2)
    else:
        major, minor = max(w, h), min(w, h)
    ar = major / max(minor, 1e-6)
    shape = "round" if ar < 1.25 else "oval" if ar < 1.6 else "oblong / elongated" if ar < 2.2 else "very elongated"
    return {
        "shape": shape,
        "n_fruit_regions": len([b for b in big if cv2.contourArea(b) >= 0.1 * area]),
        "area_px": int(area),
        "length_px": round(float(major), 1),
        "width_px": round(float(minor), 1),
        "aspect_ratio": round(float(ar), 3),
        "circularity": round(float(4 * np.pi * area / perim**2), 3),
        "solidity": round(float(area / hull_area), 3),
        "extent": round(float(area / (w * h)), 3),
        "eccentricity": round(float(np.sqrt(1 - (minor / major) ** 2)), 3),
    }


def texture_features(rgb, mask):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    x, y, w, h = cv2.boundingRect(mask)
    g, m = gray[y : y + h, x : x + w], mask[y : y + h, x : x + w] > 0
    # GLCM on fruit pixels only: background set to level 0 and level 0 excluded.
    q = (g // 8 + 1).astype(np.uint8)
    q[~m] = 0
    glcm = graycomatrix(q, [1, 3], [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=33)[1:, 1:]
    glcm = glcm / np.maximum(glcm.sum(axis=(0, 1), keepdims=True), 1)
    props = {p: round(float(graycoprops(glcm, p).mean()), 4)
             for p in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]}

    lbp = local_binary_pattern(g, 8, 1, "uniform")[m]
    hist = np.bincount(lbp.astype(int), minlength=10) / max(lbp.size, 1)
    lbp_entropy = float(-(hist[hist > 0] * np.log2(hist[hist > 0])).sum())

    inner = cv2.erode(mask, np.ones((9, 9), np.uint8)) > 0
    edges = cv2.Canny(cv2.GaussianBlur(gray, (3, 3), 0), 60, 150) > 0
    wrinkle = float(edges[inner].mean()) if inner.any() else 0.0
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    specular = (hsv[..., 2] > 225) & (hsv[..., 1] < 60)
    gloss = float(specular[mask > 0].mean())
    level = lambda v, lo, hi: "low" if v < lo else "medium" if v < hi else "high"
    return {
        **{f"glcm_{k}": v for k, v in props.items()},
        "lbp_entropy": round(lbp_entropy, 3),
        "wrinkle_index": round(wrinkle, 4),
        "wrinkling": level(wrinkle, 0.04, 0.10),
        "gloss_index": round(gloss, 4),
        "gloss": level(gloss, 0.005, 0.02),
        "surface": "smooth" if props["homogeneity"] > 0.45 and wrinkle < 0.05 else "textured / wrinkled",
    }


def extract(path_or_rgb):
    rgb = load_rgb(path_or_rgb) if not isinstance(path_or_rgb, np.ndarray) else path_or_rgb
    mask = segment_fruit(rgb)
    if mask.sum() < 0.01 * 255 * mask.size:
        return {"error": "fruit not found"}, mask
    feats = {
        "fruit_fraction_of_image": round(float((mask > 0).mean()), 3),
        "color": color_features(rgb, mask),
        "shape": shape_features(mask),
        "texture": texture_features(rgb, mask),
    }
    return feats, mask


def flatten(feats):
    out = {}
    for group, d in feats.items():
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(v, (int, float)):
                    out[f"{group}.{k}"] = v
        elif isinstance(d, (int, float)):
            out[group] = d
    return out


def _profile_row(p):
    feats, _ = extract(p)
    if "error" in feats:
        return None
    return {"class": p.parent.name, "source": p.name.split("__")[0], "file": p.name, **flatten(feats),
            "color.color_name": feats["color"]["color_name"], "shape.shape": feats["shape"]["shape"]}


def class_profiles():
    from config import OUTPUT_DIR, PROCESSED_DIR

    import concurrent.futures as cf

    paths = sorted((PROCESSED_DIR / "train").glob("*/*.jpg"))
    with cf.ProcessPoolExecutor() as pool:
        rows = [r for r in pool.map(_profile_row, paths, chunksize=8) if r]
    df = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(exist_ok=True)
    df.to_csv(OUTPUT_DIR / "train_image_features.csv", index=False)
    num = df.select_dtypes("number").columns
    prof = df.groupby("class")[num].median().round(4)
    for col in ["color.color_name", "shape.shape"]:
        prof[col] = df.groupby("class")[col].agg(lambda s: s.mode().iat[0])
    prof.to_csv(OUTPUT_DIR / "class_feature_profiles.csv")
    (OUTPUT_DIR / "class_feature_profiles.json").write_text(prof.to_json(orient="index", indent=2))
    print("\n", prof.T.to_string())


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("image", nargs="?")
    ap.add_argument("--class-profiles", action="store_true")
    a = ap.parse_args()
    if a.class_profiles:
        class_profiles()
    elif a.image:
        print(json.dumps(extract(a.image)[0], indent=2))
    else:
        ap.print_help(sys.stderr)

In [ ]:
%%writefile src/grade.py
"""Full Phase 2 quality report for a photo of one or more date fruits (a single fruit,
a tray, or the top of an open bag).

Per fruit: variety (Phase 1 model), maturity stage (CNN), quality grade 1-3 (CNN
+ Grad-CAM), size class, colour uniformity, surface defects, skin condition and
estimated weight. For the whole photo: fruit count, size-class counts and total
estimated weight.

Scale: a printed ArUco marker in the photo (config.ARUCO_*) or --mm-per-px.
Without a scale, size uses the pixel thresholds fitted on the grading rig and
weight is not estimated.
--bag-weight: the weight the scale shows for the fruits in the photo. It is
used to recalibrate the weight coefficient for this batch (k = bag weight /
sum of L*W^2), which also gives a per-fruit weight share.

Usage: python src/grade.py <image> [<image> ...] [--variety Al-majdool] [--mm-per-px 0.12] [--bag-weight 250]
Python: from grade import load_models, grade_image; grade_image("photo.jpg", load_models())
"""
import argparse
import json
import sys
from pathlib import Path

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.patches  # noqa: E402
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

import config
from explain import gradcam, overlay, upsample
from features import WORK_SIDE
from quality import analyse_fruit, find_scale, fruit_closeup, fruit_instances, hide_marker

DEFECT_COLORS = {"crack": (255, 0, 0), "black_spot": (255, 0, 255), "insect_hole": (0, 255, 255),
                 "sunburn": (255, 200, 0), "mold": (0, 255, 0), "sugar_spot": (255, 255, 255)}


def load_model(runs):
    """{"members": [(model, meta), ...], "class_names", "calib"} for one run or an ensemble."""
    runs = [runs] if isinstance(runs, str) else list(runs)
    members = []
    for run in runs:
        meta_path = config.MODEL_DIR / f"{run}_meta.json"  # written when training has finished
        if not meta_path.exists():
            print(f"note: model '{run}' not trained yet, skipped", file=sys.stderr)
            return None
        members.append((tf.keras.models.load_model(config.MODEL_DIR / f"{run}_best.keras"),
                        json.loads(meta_path.read_text())))
    names = members[0][1]["class_names"]
    assert all(m["class_names"] == names for _, m in members), "ensemble members disagree on classes"
    calib_path = config.MODEL_DIR / f"{config.run_name(runs)}_calib.json"
    calib = json.loads(calib_path.read_text()) if calib_path.exists() else {}
    return {"members": members, "class_names": names, "calib": calib}


def fruit_crop(rgb, mask, margin=0.15):
    """Square crop around one fruit with the other fruits greyed out, padded like training."""
    x, y, w, h = cv2.boundingRect(mask)
    m = int(max(w, h) * margin)
    x0, y0, x1, y1 = max(0, x - m), max(0, y - m), min(rgb.shape[1], x + w + m), min(rgb.shape[0], y + h + m)
    crop = rgb[y0:y1, x0:x1].copy()
    keep = cv2.dilate(mask[y0:y1, x0:x1], np.ones((15, 15), np.uint8)) > 0
    bg = np.median(rgb[mask == 0], 0) if (mask == 0).any() else np.array([200, 200, 200])
    crop[~keep] = bg
    side = max(crop.shape[:2])
    out = np.empty((side, side, 3), np.uint8)
    out[:] = bg
    oy, ox = (side - crop.shape[0]) // 2, (side - crop.shape[1]) // 2
    out[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop
    return out


def classify(model_meta, crop):
    """Top class with temperature-calibrated probabilities (calibrate.py). A prediction
    below the model's review threshold is flagged for manual review."""
    probs, xs = [], []
    for model, meta in model_meta["members"]:
        S = meta["img_size"]
        x = tf.constant(np.asarray(Image.fromarray(crop).resize((S, S), Image.BILINEAR), np.float32)[None])
        probs.append(model(x, training=False)[0].numpy().astype(np.float64))
        xs.append(x)
    probs = np.mean(probs, axis=0)
    calib = model_meta["calib"]
    T = calib.get("temperature", 1.0)
    z = np.log(np.clip(probs, 1e-12, 1)) / T
    probs = np.exp(z - z.max())
    probs /= probs.sum()
    i = int(probs.argmax())
    thr = calib.get("review_threshold")
    names = model_meta["class_names"]
    return {"label": names[i], "confidence": round(float(probs[i]), 3),
            "needs_review": bool(thr is not None and probs[i] < thr),
            "probs": {c: round(float(p), 3) for c, p in zip(names, probs)}}, xs[0], i


def load_models(runs=None):
    """{"variety"|"grade"|"maturity": load_model(...) or None}; runs default to config.PRODUCTION_RUNS."""
    runs = {**config.PRODUCTION_RUNS, **(runs or {})}
    return {k: load_model(r) for k, r in runs.items()}


def load_image(path):
    try:
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    except FileNotFoundError:
        raise ValueError(f"image not found: {path}")
    except Exception as e:  # PIL.UnidentifiedImageError, truncated files, ...
        raise ValueError(f"cannot read image {path}: {e}")
    if min(img.size) < 64:
        raise ValueError(f"image too small ({img.width}x{img.height}); need at least 64 px")
    return img


def grade_image(image, models, variety_override=None, mm_per_px_arg=None, bag_weight=None, save=True):
    """Full quality report (dict) for one photo. Raises ValueError for unusable input."""
    if bag_weight is not None and bag_weight <= 0:
        raise ValueError("bag weight must be positive")
    if mm_per_px_arg is not None and mm_per_px_arg <= 0:
        raise ValueError("mm per pixel must be positive")
    register_heif_opener()
    full = load_image(image)
    px_scale = min(1.0, WORK_SIDE / max(full.size))  # working px per original px
    rgb = np.asarray(full.resize((round(full.width * px_scale), round(full.height * px_scale)), Image.LANCZOS))
    full = np.asarray(full)
    mm_per_px, marker = find_scale(rgb)
    if mm_per_px_arg:
        mm_per_px = mm_per_px_arg / px_scale
    rgb = hide_marker(rgb, marker)
    if marker is not None:
        full = hide_marker(full, marker / px_scale)

    masks = fruit_instances(rgb)
    if not masks:
        raise ValueError("no fruit found in the image")

    fruits, panels = [], []
    for n, mask in enumerate(masks, 1):
        close_rgb, close_mask = fruit_closeup(full, mask, px_scale)
        crop = fruit_crop(close_rgb, close_mask)
        rec = {"fruit": n}
        if variety_override:
            rec["variety"] = {"label": variety_override, "confidence": None}
        elif models["variety"]:
            rec["variety"] = classify(models["variety"], crop)[0]
        variety = rec.get("variety", {}).get("label")
        if models["maturity"]:
            rec["maturity"] = classify(models["maturity"], crop)[0]
        cam = None
        if models["grade"]:
            rec["grade"], x, gi = classify(models["grade"], crop)
            cam = upsample(gradcam(models["grade"]["members"][0][0], x, gi), x.shape[1])  # first member
            panels.append((x[0].numpy().astype(np.uint8), cam, f"#{n} {rec['grade']['label']}"))
        # The size model knows the grading-set varieties by their folder names.
        size_variety = variety if variety in {"Aseel", "Fasli Toto", "Gajar", "Kupro"} else None
        q, dmasks = analyse_fruit(rgb, mask, mm_per_px, size_variety, px_scale, (close_rgb, close_mask))
        rec.update(q)
        rec["_mask"], rec["_defects"], rec["_close"] = mask, dmasks, close_rgb
        fruits.append(rec)

    summary = {"image": str(image), "n_fruits": len(fruits),
               "scale_mm_per_px": round(mm_per_px, 4) if mm_per_px else None,
               "scale_source": "argument" if mm_per_px_arg else "aruco" if marker is not None else None}
    sizes = [f["size"].get("size_class") for f in fruits]
    summary["size_counts"] = {s: sizes.count(s) for s in sorted(set(sizes), key=str)}
    if mm_per_px:
        lw2 = np.array([f["geometry"]["length_mm"] * f["geometry"]["width_mm"] ** 2 for f in fruits])
        summary["estimated_total_weight_g"] = round(float(config.WEIGHT_K * lw2.sum()), 1)
        if bag_weight:
            k = bag_weight / lw2.sum()
            summary["bag_weight_g"] = bag_weight
            summary["calibrated_weight_k"] = round(float(k), 7)
            summary["model_vs_scale_error_pct"] = round(
                100 * (summary["estimated_total_weight_g"] - bag_weight) / bag_weight, 1)
            for f, v in zip(fruits, lw2):
                f["weight_from_bag_g"] = round(float(k * v), 2)
    elif bag_weight:
        summary["bag_weight_g"] = bag_weight
        summary["mean_weight_per_fruit_g"] = round(bag_weight / len(fruits), 2)
    for key in ("grade", "maturity", "variety"):
        labels = [f[key]["label"] for f in fruits if key in f]
        if labels:
            summary[f"{key}_counts"] = {l: labels.count(l) for l in sorted(set(labels))}
    summary["fruits_with_defects"] = sum(bool(f["surface_defects"].get("defects_present")) for f in fruits)
    summary["fruits_needing_review"] = [f["fruit"] for f in fruits
                                        if any(f.get(k, {}).get("needs_review") for k in ("variety", "grade", "maturity"))]
    report = {"summary": summary, "fruits": [{k: v for k, v in f.items() if not k.startswith("_")} for f in fruits]}
    if not save:
        return report

    out = config.OUTPUT_DIR / "quality" / "reports"
    out.mkdir(parents=True, exist_ok=True)
    stem = Path(str(image)).stem
    annotated = rgb.copy()
    for f in fruits:
        cnts, _ = cv2.findContours(f["_mask"], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(annotated, cnts, -1, (0, 200, 0), 2)
        x, y, _, _ = cv2.boundingRect(f["_mask"])
        cv2.putText(annotated, str(f["fruit"]), (x + 4, y + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(annotated, str(f["fruit"]), (x + 4, y + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    # Column 0: overview. Then per fruit (max 5): defect candidates (top), grade Grad-CAM (bottom).
    shown = fruits[:5]
    fig, axes = plt.subplots(2, 1 + len(shown), figsize=(4 * (1 + len(shown)), 8), squeeze=False)
    axes[0, 0].imshow(annotated)
    axes[0, 0].set_title(f"{len(fruits)} fruit(s)")
    axes[1, 0].legend(handles=[matplotlib.patches.Patch(facecolor=np.array(c) / 255, edgecolor="k", label=k)
                               for k, c in DEFECT_COLORS.items()], loc="center", title="defect candidates")
    cams = {int(t.split()[0][1:]): (img, cam) for img, cam, t in panels}
    for col, f in enumerate(shown, 1):
        img = f["_close"].copy()
        for name, m in f["_defects"].items():
            img[m] = (0.3 * img[m] + 0.7 * np.array(DEFECT_COLORS[name])).astype(np.uint8)
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"#{f['fruit']} " + (", ".join(f["surface_defects"].get("defects_present", []))
                                                    or "no defects"), fontsize=9)
        if f["fruit"] in cams:
            axes[1, col].imshow(overlay(*cams[f["fruit"]]))
            g = f["grade"]
            axes[1, col].set_title(f"grade Grad-CAM: {g['label']} ({g['confidence']:.2f})", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(out / f"{stem}_quality.png", dpi=100)

    plt.close(fig)
    (out / f"{stem}_quality.json").write_text(json.dumps(report, indent=2))
    return report


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("images", nargs="+")
    ap.add_argument("--variety", help="skip the variety model and use this variety")
    ap.add_argument("--mm-per-px", type=float, help="mm per pixel of the ORIGINAL image")
    ap.add_argument("--bag-weight", type=float, help="scale reading (g) for the fruits in the photo")
    ap.add_argument("--variety-run", nargs="+")
    ap.add_argument("--grade-run", nargs="+")
    ap.add_argument("--maturity-run", nargs="+")
    args = ap.parse_args()
    runs = {k: v for k, v in [("variety", args.variety_run), ("grade", args.grade_run),
                              ("maturity", args.maturity_run)] if v}
    models = load_models(runs)
    failed = 0
    for image in args.images:
        try:
            report = grade_image(image, models, args.variety, args.mm_per_px, args.bag_weight)
            print(json.dumps(report, indent=2))
        except ValueError as e:
            failed += 1
            print(json.dumps({"image": image, "error": str(e)}), file=sys.stderr)
    sys.exit(1 if failed else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/model.py
"""Transfer-learning classifier: ImageNet backbone ("backbone") + small head ("head").

The two parts are kept as named sub-models so explain.py can split the forward
pass at the last convolutional feature map for Grad-CAM. Every backbone here
takes raw [0, 255] pixels (preprocessing is built in)."""
import tensorflow as tf

A = tf.keras.applications
# name -> (constructor, default input size)
BACKBONES = {
    "mobilenetv3": (lambda **kw: A.MobileNetV3Large(include_preprocessing=True, **kw), 224),
    "efficientnet": (A.EfficientNetB0, 224),  # closest Keras counterpart of EfficientNet-Lite
    "efficientnetv2b0": (lambda **kw: A.EfficientNetV2B0(include_preprocessing=True, **kw), 260),
    "efficientnetv2s": (lambda **kw: A.EfficientNetV2S(include_preprocessing=True, **kw), 384),
    "convnext_tiny": (lambda **kw: A.ConvNeXtTiny(include_preprocessing=True, **kw), 288),
    "convnext_small": (lambda **kw: A.ConvNeXtSmall(include_preprocessing=True, **kw), 288),
}


def build_backbone(name, img_size):
    base = BACKBONES[name][0](input_shape=(img_size, img_size, 3), include_top=False, weights="imagenet")
    return tf.keras.Model(base.input, base.output, name="backbone")


def build_model(num_classes, backbone_name, img_size, dropout=0.3):
    backbone = build_backbone(backbone_name, img_size)
    backbone.trainable = False
    head = tf.keras.Sequential(
        [
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(num_classes, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            # float32 softmax so mixed precision stays numerically safe
            tf.keras.layers.Activation("softmax", dtype="float32"),
        ],
        name="head",
    )
    inputs = tf.keras.Input((img_size, img_size, 3), name="image")
    # training=False keeps BatchNorm statistics frozen, also during fine-tuning.
    outputs = head(backbone(inputs, training=False))
    return tf.keras.Model(inputs, outputs, name=f"date_classifier_{backbone_name}")


def unfreeze_top(model, n_layers):
    """Unfreeze the last n_layers of the backbone (n_layers <= 0: all). BatchNorm stays frozen."""
    backbone = model.get_layer("backbone")
    backbone.trainable = True
    if n_layers > 0:
        for layer in backbone.layers[:-n_layers]:
            layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

In [ ]:
%%writefile src/predict.py
"""Classify one date-fruit image and report what was extracted from it:
predicted variety (top-3), Grad-CAM heatmap, and colour/shape/texture features
compared with the predicted variety's typical profile.

Usage: python src/predict.py <image> [--run effv2b0_260]   (default: the production variety model)
"""
import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

import config
from explain import gradcam, overlay, upsample
from features import extract, load_rgb

COMPARE = ["color.mean_hue_deg", "color.mean_brightness", "color.color_uniformity",
           "shape.aspect_ratio", "shape.circularity", "texture.glcm_homogeneity", "texture.wrinkle_index"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("image")
    ap.add_argument("--run", default=config.PRODUCTION_RUNS["variety"][0])
    args = ap.parse_args()

    model = tf.keras.models.load_model(config.MODEL_DIR / f"{args.run}_best.keras")
    meta = json.loads((config.MODEL_DIR / f"{args.run}_meta.json").read_text())
    class_names = meta["class_names"]

    rgb = load_rgb(args.image)
    S = meta.get("img_size", config.IMG_SIZE)
    h, w = rgb.shape[:2]
    side = min(h, w)  # centre crop, same as training
    crop = rgb[(h - side) // 2 : (h + side) // 2, (w - side) // 2 : (w + side) // 2]
    img = np.asarray(Image.fromarray(crop).resize((S, S), Image.BILINEAR)).astype(np.float32)
    x = tf.constant(img[None])
    probs = model(x, training=False)[0].numpy().astype(np.float64)
    calib_path = config.MODEL_DIR / f"{args.run}_calib.json"  # temperature from calibrate.py
    if calib_path.exists():
        calib = json.loads(calib_path.read_text())
        z = np.log(np.clip(probs, 1e-12, 1)) / calib["temperature"]
        probs = np.exp(z - z.max())
        probs /= probs.sum()
    top = probs.argsort()[::-1][:3]
    pred = class_names[top[0]]

    feats, mask = extract(rgb)
    report = {
        "image": args.image,
        "predicted_variety": pred,
        "confidence": round(float(probs[top[0]]), 4),
        "needs_review": bool(calib_path.exists() and probs[top[0]] < (calib.get("review_threshold") or 0)),
        "top3": [{"variety": class_names[i], "prob": round(float(probs[i]), 4)} for i in top],
        "features": feats,
    }
    prof_path = config.OUTPUT_DIR / "class_feature_profiles.csv"
    if prof_path.exists() and "error" not in feats:
        prof = pd.read_csv(prof_path, index_col=0)
        if pred in prof.index:
            flat = {f"{g}.{k}": v for g, d in feats.items() if isinstance(d, dict) for k, v in d.items()}
            report["vs_typical_" + pred] = {
                k: {"this_image": flat[k], "variety_median": float(prof.loc[pred, k])} for k in COMPARE if k in prof
            }

    out = config.OUTPUT_DIR / "predictions"
    out.mkdir(parents=True, exist_ok=True)
    stem = Path(args.image).stem
    cam = upsample(gradcam(model, x, int(top[0])), S)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, panel, title in zip(axes, [img.astype(np.uint8), overlay(img, cam), mask],
                                ["input", f"Grad-CAM → {pred} ({probs[top[0]]:.2f})", "fruit mask (features)"]):
        ax.imshow(panel, cmap="gray" if panel is mask else None)
        ax.set_title(title), ax.axis("off")
    fig.tight_layout()
    fig.savefig(out / f"{stem}_explained.png", dpi=110)
    (out / f"{stem}_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/prepare_data.py
"""Build the Phase 1 dataset from both sources and write a stratified
train/val/test split to data/processed/<split>/<class>/.

* Zenodo 4639543: filename prefix -> class; HEIC converted; EXIF rotation fixed.
* Kaggle Alhamdan: one fruit on a plain grey background, ~15% of a 5184x3456
  frame, so each image is auto-cropped to a square around the fruit.

Stratification is on class+source, so every split holds both photo styles of a
variety that has both.
"""
import concurrent.futures as cf
import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
from sklearn.model_selection import train_test_split

from config import (ALHAMDAN_DIR, ALHAMDAN_MAP, CLASSES, PROCESSED_DIR, RAW_DIR, SEED, SPLIT, SPLITS_CSV,
                    STORE_MAX_SIDE, ZENODO_MAP)

register_heif_opener()
EXTS = {".jpg", ".jpeg", ".png", ".heic"}


def zenodo_class(filename):
    return ZENODO_MAP.get(re.sub(r"[\s_\-\(]*\d.*$", "", filename))


def collect():
    rows = [{"path": str(p), "label": zenodo_class(p.name), "source": "zenodo"}
            for p in sorted(RAW_DIR.iterdir()) if p.suffix.lower() in EXTS and zenodo_class(p.name)]
    for folder, label in ALHAMDAN_MAP.items():
        rows += [{"path": str(p), "label": label, "source": "alhamdan"}
                 for p in sorted((ALHAMDAN_DIR / folder).iterdir()) if p.suffix.lower() in EXTS]
    df = pd.DataFrame(rows)
    return df[df.label.isin(CLASSES)].reset_index(drop=True)


def crop_to_fruit(img, margin=0.3):
    """Square crop around the fruit on a plain background."""
    small = np.asarray(img.resize((img.width // 4, img.height // 4)))
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).astype(np.float32)
    border = np.concatenate([lab[:8].reshape(-1, 3), lab[-8:].reshape(-1, 3),
                             lab[:, :8].reshape(-1, 3), lab[:, -8:].reshape(-1, 3)])
    dist = np.linalg.norm(lab - np.median(border, 0), axis=2)
    dist = cv2.GaussianBlur(np.clip(dist * 4, 0, 255).astype(np.uint8), (5, 5), 0)
    _, mask = cv2.threshold(dist, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    H, W = mask.shape
    # Ignore blobs touching the frame edge (vignetting / table shadow), keep the biggest other one.
    inner = [i for i in range(1, n) if stats[i, 0] > 0 and stats[i, 1] > 0
             and stats[i, 0] + stats[i, 2] < W and stats[i, 1] + stats[i, 3] < H]
    if not inner:
        return img
    x, y, w, h, _ = stats[max(inner, key=lambda i: stats[i, cv2.CC_STAT_AREA])] * 4
    side = min(int(max(w, h) * (1 + 2 * margin)), img.width, img.height)
    cx, cy = x + w // 2, y + h // 2
    left = int(np.clip(cx - side // 2, 0, max(img.width - side, 0)))
    top = int(np.clip(cy - side // 2, 0, max(img.height - side, 0)))
    return img.crop((left, top, left + side, top + side))


def process(row):
    src, out, source = row
    img = Image.open(src)
    if source == "alhamdan":
        img.draft("RGB", (img.width // 2, img.height // 2))
    img = ImageOps.exif_transpose(img).convert("RGB")
    if source == "alhamdan":
        img = crop_to_fruit(img)
    img.thumbnail((STORE_MAX_SIDE, STORE_MAX_SIDE), Image.LANCZOS)
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out, quality=95)


def main():
    df = collect()
    too_few = df.label.value_counts().loc[lambda s: s < 20]
    if len(too_few):
        raise SystemExit(f"Too few images for {dict(too_few)}: is the Zenodo download in {RAW_DIR} complete?")
    strata = df.label + "|" + df.source
    # A class+source group too small to spread over three splits is stratified by class only.
    counts = strata.map(strata.value_counts())
    strata = strata.where(counts >= 10, df.label)
    train, rest = train_test_split(df, train_size=SPLIT["train"], stratify=strata, random_state=SEED)
    val_frac = SPLIT["val"] / (SPLIT["val"] + SPLIT["test"])
    val, test = train_test_split(rest, train_size=val_frac, stratify=strata[rest.index], random_state=SEED)
    df = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
    df["out"] = [
        str(PROCESSED_DIR / r.split / r.label / f"{r.source}__{Path(r.path).stem}.jpg")
        for r in df.itertuples()
    ]

    if PROCESSED_DIR.exists():
        shutil.rmtree(PROCESSED_DIR)
    jobs = [(r.path, Path(r.out), r.source) for r in df.itertuples()]
    with cf.ProcessPoolExecutor() as pool:
        for i, _ in enumerate(pool.map(process, jobs, chunksize=8), 1):
            if i % 200 == 0:
                print(f"{i}/{len(jobs)}", flush=True)
    df.to_csv(SPLITS_CSV, index=False)
    print(pd.crosstab([df.label, df.source], df.split, margins=True).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/prepare_quality.py
"""Build the Phase 2 datasets under data/quality/.

grade/<split>/Grade-k/     Kaggle variety/size/grade set: 3,004 single-fruit crops of 4
                           Pakistani varieties (Aseel, Fasli Toto, Gajar, Kupro). Split
                           70/15/15, stratified on variety+size+grade.
maturity/<split>/<stage>/  Immature / Khalal / Rutab / Tamar.
                           * Zenodo 10143465 (Moroccan orchards): bunch crops from
                             scripts/download_maturity.py. Each stage is one photo session per
                             variety, so the whole MATURITY_TEST_VARIETY goes to test and a
                             random split is used only for train/val.
                           * Alhamdan studio photos: the "Rutab" folder, and an equal number of
                             Tamar fruits sampled from the other varieties. Single fruits on a
                             plain background, the way a dealer photographs them.
grading_index.csv          one row per grading-set fruit: variety, size, grade, split and the
                           fruit's length/width/area in pixels (for the size model in quality.py).

Every image is padded (not cropped) to a square, so no part of the fruit is cut
off. Filenames start with "<source-or-variety>__", which evaluate.py reports
accuracy by.

Usage: python src/prepare_quality.py [--only grade|maturity]
"""
import argparse
import concurrent.futures as cf
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

from config import (ALHAMDAN_DIR, GRADING_DIR, MATURITY_SRC_DIR, MATURITY_TEST_VARIETY, QUALITY_DIR, SEED, SPLIT,
                    STORE_MAX_SIDE)
from features import segment_fruit
from prepare_data import crop_to_fruit

TAMAR_FOLDERS = ["Ajwa", "Medjool", "Sokari", "Sugaey", "Nabtat Ali", "Meneifi", "Shaishe", "Galaxy"]


def pad_square(img):
    """Pad to a square with the median border colour."""
    a = np.asarray(img)
    border = np.concatenate([a[:4].reshape(-1, 3), a[-4:].reshape(-1, 3), a[:, :4].reshape(-1, 3), a[:, -4:].reshape(-1, 3)])
    fill = tuple(int(c) for c in np.median(border, 0))
    side = max(img.size)
    out = Image.new("RGB", (side, side), fill)
    out.paste(img, ((side - img.width) // 2, (side - img.height) // 2))
    return out


def measure(img):
    """Fruit length / width / area in pixels of the original crop."""
    rgb = cv2.copyMakeBorder(np.asarray(img), 40, 40, 40, 40, cv2.BORDER_REPLICATE)
    mask = segment_fruit(rgb)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cnts, key=cv2.contourArea)
    (_, _), (a, b), _ = cv2.minAreaRect(c)
    return {"length_px": max(a, b), "width_px": min(a, b), "area_px": cv2.contourArea(c)}


def process(job):
    src, out, kind = job
    img = ImageOps.exif_transpose(Image.open(src))
    if kind == "alhamdan":
        img.draft("RGB", (img.width // 2, img.height // 2))
        img = crop_to_fruit(img.convert("RGB"))
    img = img.convert("RGB")
    geom = measure(img) if kind == "grading" else {}
    img = pad_square(img)
    img.thumbnail((STORE_MAX_SIDE, STORE_MAX_SIDE), Image.LANCZOS)
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out, quality=95)
    return geom


def split3(df, strata):
    train, rest = train_test_split(df, train_size=SPLIT["train"], stratify=strata, random_state=SEED)
    val_frac = SPLIT["val"] / (SPLIT["val"] + SPLIT["test"])
    val, test = train_test_split(rest, train_size=val_frac, stratify=strata[rest.index], random_state=SEED)
    return pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])


def grading_rows():
    rows = [{"path": str(p), "variety": p.parts[-4], "size": p.parts[-3], "grade": p.parts[-2]}
            for p in sorted(GRADING_DIR.glob("*/*/*/*")) if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    df = pd.DataFrame(rows)
    df = split3(df, df.variety + "|" + df["size"] + "|" + df.grade)
    df["out"] = [str(QUALITY_DIR / "grade" / r.split / r.grade /
                     f"{r.variety.replace(' ', '-')}__{r.size}__{Path(r.path).stem}.jpg") for r in df.itertuples()]
    df["kind"] = "grading"
    return df


def maturity_rows():
    crops = MATURITY_SRC_DIR / "crops"
    mor = pd.DataFrame([{"path": str(p), "stage": p.parent.name, "variety": p.name.split("__")[0]}
                        for p in sorted(crops.glob("*/*.jpg"))])
    if mor.empty:
        raise SystemExit(f"No crops in {crops}: run scripts/download_maturity.py first")
    test = mor[mor.variety == MATURITY_TEST_VARIETY].assign(split="test")
    rest = mor[mor.variety != MATURITY_TEST_VARIETY]
    val_frac = SPLIT["val"] / (SPLIT["train"] + SPLIT["val"])
    tr, va = train_test_split(rest, test_size=val_frac, stratify=rest.variety + rest.stage, random_state=SEED)
    mor = pd.concat([tr.assign(split="train"), va.assign(split="val"), test])
    mor["source"] = "morocco-" + mor.variety
    mor["kind"] = "morocco"

    rutab = sorted((ALHAMDAN_DIR / "Rutab").iterdir())
    rng = np.random.default_rng(SEED)
    per = int(np.ceil(len(rutab) / len(TAMAR_FOLDERS)))
    tamar = [p for f in TAMAR_FOLDERS for p in rng.choice(sorted((ALHAMDAN_DIR / f).iterdir()), per, replace=False)]
    alh = pd.DataFrame([{"path": str(p), "stage": "Rutab"} for p in rutab] +
                       [{"path": str(p), "stage": "Tamar"} for p in tamar])
    alh = split3(alh, alh.stage).assign(source="alhamdan", kind="alhamdan")

    df = pd.concat([mor, alh], ignore_index=True)
    df["out"] = [str(QUALITY_DIR / "maturity" / r.split / r.stage / f"{r.source}__{Path(r.path).stem}.jpg")
                 for r in df.itertuples()]
    return df


def run(df):
    jobs = [(r.path, Path(r.out), r.kind) for r in df.itertuples()]
    with cf.ProcessPoolExecutor() as pool:
        return list(pool.map(process, jobs, chunksize=16))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--only", choices=["grade", "maturity"])
    args = ap.parse_args()
    for sub in ["grade", "maturity"]:
        if args.only in (None, sub) and (QUALITY_DIR / sub).exists():
            shutil.rmtree(QUALITY_DIR / sub)

    if args.only in (None, "grade"):
        g = grading_rows()
        g = pd.concat([g.reset_index(drop=True), pd.DataFrame(run(g))], axis=1)
        g.drop(columns=["kind"]).to_csv(QUALITY_DIR / "grading_index.csv", index=False)
        print(pd.crosstab([g.variety, g.grade], g.split, margins=True).to_string(), "\n")

    if args.only in (None, "maturity"):
        m = maturity_rows()
        run(m)
        m.drop(columns=["kind"]).to_csv(QUALITY_DIR / "maturity_index.csv", index=False)
        print(pd.crosstab([m.source, m.stage], m.split, margins=True).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/quality.py
"""Phase 2 per-fruit quality estimators that need no training: scale, size, colour
uniformity, surface defects, skin condition and weight. The two learned
estimators (quality grade, maturity stage) are CNNs trained with train.py;
grade.py puts everything together.

The defect and skin detectors are colour/texture rules, not trained models: no
public dataset labels date defects by type. They are checked indirectly
(`--check-defects`: defect area should rise from Grade-1 to Grade-3 in the
grading set) and should be tuned on dealer photos with labelled defects.

Usage:
  python src/quality.py <image>                  # quality features of the largest fruit
  python src/quality.py --fit-size               # fit + test size thresholds on the grading set
  python src/quality.py --check-defects          # defect/skin statistics by grade (sanity check)
  python src/quality.py --calibrate-weight w.csv # csv: image,weight_g[,mm_per_px] -> fitted WEIGHT_K
"""
import argparse
import json
import os
import sys

import cv2
import joblib
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

import config
from features import color_features, load_rgb, segment_fruit, texture_features

SIZE_MODEL = config.OUTPUT_DIR / "quality" / "size_model.json"  # {"mm": {variety: {"metric", "cuts"}}}
SIZE_FOREST = config.OUTPUT_DIR / "quality" / "size_forest.joblib"
SIZE_FEATURES = ["sqrt_area_px", "length_px", "width_px"]
SIZES = ["Small", "Medium", "Large"]
# Fallback length thresholds (mm) when a variety has no fitted model: typical trade
# ranges for mid-sized dates. Replace with the dealer's own grading rules.
DEFAULT_SIZE_MM = {"metric": "length_mm", "cuts": [30.0, 38.0]}


# ---------------------------------------------------------------- scale & instances
def find_scale(rgb):
    """(mm per pixel, marker corners) from a printed ArUco marker (config.ARUCO_*), or (None, None)."""
    aruco = cv2.aruco
    det = aruco.ArucoDetector(aruco.getPredefinedDictionary(getattr(aruco, config.ARUCO_DICT)))
    corners, ids, _ = det.detectMarkers(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY))
    if ids is None:
        return None, None
    c = corners[0][0]
    side_px = np.mean([np.linalg.norm(c[i] - c[(i + 1) % 4]) for i in range(4)])
    return float(config.ARUCO_SIDE_MM / side_px), c


def hide_marker(rgb, corners):
    """Paint the marker (and its white quiet zone) with the background colour so it is
    not segmented as a fruit."""
    if corners is None:
        return rgb
    centre = corners.mean(0)
    poly = np.int32(centre + (corners - centre) * 1.6)
    out = rgb.copy()
    border = np.concatenate([rgb[:6].reshape(-1, 3), rgb[-6:].reshape(-1, 3)])
    cv2.fillConvexPoly(out, poly, tuple(int(v) for v in np.median(border, 0)))
    return out


def _neck_groups(dist, peaks, neck_ratio=0.6):
    """Group distance-transform peaks that belong to the same fruit. Two peaks are
    separate fruits only if the fruit's half-width (the distance value) drops below
    neck_ratio x the smaller peak somewhere on the straight line between them.
    A single elongated date has a flat ridge, so its peaks stay together."""
    parent = list(range(len(peaks)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i in range(len(peaks)):
        for j in range(i + 1, len(peaks)):
            (r0, c0), (r1, c1) = peaks[i], peaks[j]
            n = int(max(abs(r1 - r0), abs(c1 - c0))) + 1
            line = dist[np.linspace(r0, r1, n).round().astype(int), np.linspace(c0, c1, n).round().astype(int)]
            if line.min() >= neck_ratio * min(dist[r0, c0], dist[r1, c1]):
                parent[find(i)] = find(j)
    roots = sorted({find(i) for i in range(len(peaks))})
    return [roots.index(find(i)) + 1 for i in range(len(peaks))]


def fruit_instances(rgb, min_frac=0.15):
    """One mask per fruit. Touching fruits are split with a watershed on the distance
    transform, seeded with one marker per fruit (peaks separated by a neck)."""
    mask = segment_fruit(rgb)
    n, labels = cv2.connectedComponents(mask)
    masks = []
    for i in range(1, n):
        comp = (labels == i).astype(np.uint8) * 255
        dist = ndi.distance_transform_edt(comp)
        peaks = peak_local_max(dist, min_distance=max(3, int(dist.max() * 0.5)), threshold_abs=dist.max() * 0.4,
                               labels=comp > 0, exclude_border=False)
        groups = _neck_groups(dist, peaks) if len(peaks) > 1 else [1]
        if max(groups) == 1:
            masks.append(comp)
            continue
        markers = np.zeros(comp.shape, np.int32)
        for (r, c), g in zip(peaks, groups):
            markers[r, c] = g
        ws = watershed(-dist, markers, mask=comp > 0)
        masks += [(ws == j).astype(np.uint8) * 255 for j in range(1, ws.max() + 1)]
    if not masks:
        return []
    areas = [m.sum() for m in masks]
    masks = [m for m, a in zip(masks, areas) if a >= min_frac * max(areas)]
    return sorted(masks, key=lambda m: -m.sum())


def fruit_closeup(full_rgb, mask, px_scale, margin=0.12):
    """High-resolution view of one fruit for colour/defect/skin analysis: crop the
    original image around the fruit, resize to features.WORK_SIDE and re-segment."""
    from features import WORK_SIDE

    x, y, w, h = cv2.boundingRect(mask)
    m = int(max(w, h) * margin)
    x0, y0 = max(0, int((x - m) / px_scale)), max(0, int((y - m) / px_scale))
    x1 = min(full_rgb.shape[1], int((x + w + m) / px_scale))
    y1 = min(full_rgb.shape[0], int((y + h + m) / px_scale))
    crop = full_rgb[y0:y1, x0:x1]
    s = min(1.0, WORK_SIDE / max(crop.shape[:2]))
    crop = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else crop.copy()
    # keep only this fruit: other fruits inside the box are masked by the overview mask
    own = cv2.resize(mask[max(0, y - m):y + h + m, max(0, x - m):x + w + m], crop.shape[1::-1],
                     interpolation=cv2.INTER_NEAREST)
    own = cv2.dilate(own, np.ones((9, 9), np.uint8))
    close = segment_fruit(cv2.copyMakeBorder(crop, 20, 20, 20, 20, cv2.BORDER_REPLICATE))[20:-20, 20:-20]
    return crop, cv2.bitwise_and(close, own)


# ---------------------------------------------------------------- size & weight
def geometry(mask, mm_per_px=None, px_scale=1.0):
    """Length/width/area of the fruit. px_scale = working-image px per original-image px
    (features.load_rgb resizes to 512 px): pixel values are reported in original pixels,
    which is what the size model was fitted on. mm_per_px is per working-image pixel."""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cnts, key=cv2.contourArea)
    (_, _), (a, b), _ = cv2.minAreaRect(c)
    L, W, area = max(a, b), min(a, b), cv2.contourArea(c)
    g = {"length_px": round(L / px_scale, 1), "width_px": round(W / px_scale, 1), "area_px": int(area / px_scale**2)}
    if mm_per_px:
        g.update(length_mm=round(L * mm_per_px, 1), width_mm=round(W * mm_per_px, 1),
                 area_mm2=round(area * mm_per_px**2, 1))
    return g


def size_class(geom, variety=None):
    """Small/Medium/Large. With a mm scale: the variety's mm cut points (DEFAULT_SIZE_MM
    until the dealer sets their own). Without: the variety's random forest fitted on
    the grading set, valid only for photos from that camera rig."""
    if "length_mm" in geom:
        cuts = json.loads(SIZE_MODEL.read_text()).get("mm", {}) if SIZE_MODEL.exists() else {}
        m = cuts.get(variety, DEFAULT_SIZE_MM)
        k = int(np.searchsorted(m["cuts"], geom[m["metric"]]))
        return {"size_class": SIZES[min(k, 2)], "method": f"{m['metric']} cut points", "cuts": m["cuts"]}
    forests = joblib.load(SIZE_FOREST) if SIZE_FOREST.exists() else {}
    if variety not in forests:
        return {"size_class": None, "note": "no scale reference and no pixel model for this variety"}
    f = forests[variety]
    if isinstance(f, str):  # variety with a single size class in the grading set
        return {"size_class": f, "method": "only class seen for this variety"}
    x = pd.DataFrame([[np.sqrt(geom["area_px"]), geom["length_px"], geom["width_px"]]], columns=SIZE_FEATURES)
    probs = f.predict_proba(x)[0]
    return {"size_class": str(f.classes_[probs.argmax()]), "confidence": round(float(probs.max()), 3),
            "method": "random forest on sqrt(area), length, width (grading-rig pixels)"}


def estimate_weight(geom, k=config.WEIGHT_K):
    """Ellipsoid-of-revolution weight: k * L * W^2 (mm -> g). Needs a scale."""
    if "length_mm" not in geom:
        return None
    return round(k * geom["length_mm"] * geom["width_mm"] ** 2, 2)


def fit_size():
    """Per-variety random forest on sqrt(area), length and width in pixels, trained on
    the grading set's train split and scored on its test split. (Chosen on the val
    split over logistic regression, gradient boosting and area-only cut points.)"""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import confusion_matrix

    df = pd.read_csv(config.QUALITY_DIR / "grading_index.csv")
    df["sqrt_area_px"] = np.sqrt(df.area_px)
    forests, report = {}, {}
    for variety, g in df.groupby("variety"):
        tr, te = g[g.split == "train"], g[g.split == "test"]
        if g["size"].nunique() == 1:
            forests[variety] = str(g["size"].iat[0])
            continue
        f = RandomForestClassifier(300, min_samples_leaf=5, random_state=config.SEED).fit(tr[SIZE_FEATURES], tr["size"])
        forests[variety] = f
        pred = f.predict(te[SIZE_FEATURES])
        labels = [s for s in SIZES if s in f.classes_]
        report[variety] = {"test_accuracy": round(float((pred == te["size"]).mean()), 3), "n_test": len(te),
                           "labels": labels, "confusion": confusion_matrix(te["size"], pred, labels=labels).tolist()}
    n = sum(r["n_test"] for r in report.values())
    report["overall_test_accuracy"] = round(sum(r["test_accuracy"] * r["n_test"] for r in report.values()) / n, 3)
    SIZE_FOREST.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(forests, SIZE_FOREST)
    if not SIZE_MODEL.exists():
        SIZE_MODEL.write_text(json.dumps({"mm": {}}, indent=2))
    (SIZE_MODEL.parent / "size_model_test.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


# ---------------------------------------------------------------- colour uniformity
def color_uniformity(rgb, mask):
    c = color_features(rgb, mask)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)[mask > 0].astype(float)
    lab[:, 0] *= 100 / 255
    lab[:, 1:] -= 128
    de = np.linalg.norm(lab - np.median(lab, 0), axis=1)
    within = float((de < 10).mean())  # share of the skin within ΔE 10 of its median colour
    score = c["color_uniformity"]
    return {
        "uniformity_score": score,  # exp(-mean ΔE / 15): 1 = perfectly even colour
        "mean_deltaE_from_mean": c["color_spread_deltaE"],
        "share_within_deltaE10": round(within, 3),
        "lightness_std": round(float(lab[:, 0].std()), 2),
        "level": "uniform" if score >= 0.55 else "moderately uniform" if score >= 0.4 else "non-uniform",
        "color_name": c["color_name"],
        "dominant_colors": c["dominant_colors"],
    }


# ---------------------------------------------------------------- defects & skin
def _lab(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, a, b = lab[..., 0] * 100 / 255, lab[..., 1] - 128, lab[..., 2] - 128
    return L, a, b, np.hypot(a, b)


def _local_median(ch, mask, k):
    """Median of the fruit's own pixels around each point (background filled first)."""
    filled = np.where(mask, ch, np.median(ch[mask])).astype(np.float32)
    lo, hi = filled.min(), filled.max()
    u8 = np.uint8(255 * (filled - lo) / max(hi - lo, 1e-6))
    return cv2.medianBlur(u8, k).astype(np.float32) / 255 * (hi - lo) + lo


def _blobs(bin_mask, fruit_area, min_frac, max_frac=1.0):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(bin_mask.astype(np.uint8), connectivity=8)
    out = []
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if not (min_frac * fruit_area <= area <= max_frac * fruit_area):
            continue
        comp = (lab == i).astype(np.uint8)
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        c = cnts[0]
        perim = max(cv2.arcLength(c, True), 1)
        (_, _), (w, h), _ = cv2.minAreaRect(c)
        out.append({"mask": comp > 0, "area": int(area), "circularity": 4 * np.pi * area / perim**2,
                    "elongation": max(w, h) / max(min(w, h), 1), "length": max(w, h),
                    "bbox": [int(v) for v in stats[i, :4]]})
    return out


def surface_defects(rgb, mask):
    """Rule-based candidates for cracks, black spots, sunburn, mold and insect holes.
    Thresholds are relative to the fruit's own colour so dark and light varieties
    are treated alike. Returns (summary dict, {defect: bool mask})."""
    fruit = cv2.erode(mask, np.ones((7, 7), np.uint8)) > 0  # skip the rim (shadows / background bleed)
    A = int(fruit.sum())
    if A < 500:
        return {"error": "fruit too small"}, {}
    L, a, b, C = _lab(rgb)
    side = max(rgb.shape[:2])
    k = (int(side * 0.08) | 1)  # neighbourhood ~8% of the image
    Lm, Cm = _local_median(L, fruit, k), _local_median(C, fruit, k)
    Lmed, Cmed = float(np.median(L[fruit])), float(np.median(C[fruit]))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    V = hsv[..., 2].astype(np.float32)
    Vf = V[fruit]
    # Highlights relative to the fruit's own brightness: on a dark glossy date a
    # reflection is grey, not white, and would otherwise read as mold.
    hl_thr = max(np.median(Vf) + 3 * (np.percentile(Vf, 75) - np.percentile(Vf, 25)), np.percentile(Vf, 93))
    specular = ((V >= hl_thr) | ((V > 215) & (hsv[..., 1] < 70))).astype(np.uint8)
    specular = cv2.dilate(specular, np.ones((7, 7), np.uint8)) > 0
    # Light defects are also ignored along the rim, where the skin curves away and catches light.
    rim = max(7, int(0.06 * np.sqrt(A))) | 1
    inner = cv2.erode(mask, np.ones((rim, rim), np.uint8)) > 0

    dark = fruit & (L < Lm - 12) & (L < Lmed - 10)
    dark = cv2.morphologyEx(dark.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)) > 0
    light = inner & ~specular & (L > 40)
    pale = light & (L > Lm + 12) & (C < Cm * 0.8)
    grey = light & (C < max(6.0, 0.45 * Cmed)) & (L > Lmed + 8)
    greenish = inner & (a < -4) & (b > 0) & (L > 20)

    found = {k: [] for k in ["crack", "black_spot", "insect_hole", "sunburn", "mold", "sugar_spot"]}
    for bl in _blobs(dark, A, 0.0004, 0.2):
        if bl["elongation"] > 4 and bl["length"] > 0.12 * np.sqrt(A):
            found["crack"].append(bl)
        elif bl["circularity"] > 0.65 and bl["area"] < 0.006 * A:
            found["insect_hole"].append(bl)
        elif bl["area"] >= 0.0015 * A:
            found["black_spot"].append(bl)
    # Light, desaturated regions: large smooth ones = sunburn, textured/greenish = mold,
    # many small granular ones = sugar spots (a skin condition, reported with skin).
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    local_std = np.sqrt(np.maximum(cv2.blur(gray**2, (7, 7)) - cv2.blur(gray, (7, 7)) ** 2, 0))
    tex_med = float(np.median(local_std[fruit]))
    for bl in _blobs(pale | grey | greenish, A, 0.0003, 0.6):
        m = bl["mask"]
        textured = local_std[m].mean() > 1.5 * tex_med
        green = greenish[m].mean() > 0.3
        if bl["area"] < 0.004 * A:
            found["sugar_spot"].append(bl)
        elif bl["elongation"] > 3:  # thin light strips are lit crease edges, not a defect
            continue
        elif green or (grey[m].mean() > 0.5 and textured):
            found["mold"].append(bl)
        elif bl["area"] >= 0.03 * A and not textured:
            found["sunburn"].append(bl)
        elif grey[m].mean() > 0.5:
            found["sugar_spot"].append(bl)

    masks, summary = {}, {}
    for name, blobs in found.items():
        m = np.zeros(mask.shape, bool)
        for bl in blobs:
            m |= bl["mask"]
        masks[name] = m
        summary[name] = {"count": len(blobs), "area_pct": round(100 * float(m.sum()) / A, 2)}
    # A trace amount is usually noise; these minimums decide "present".
    min_pct = {"crack": 0.3, "black_spot": 0.5, "insect_hole": 0.05, "sunburn": 3.0, "mold": 2.0, "sugar_spot": 0.3}
    for name, s in summary.items():
        s["present"] = s["area_pct"] >= min_pct[name]
    defects = {k: v for k, v in summary.items() if k != "sugar_spot"}
    return {
        "defects": defects,
        "defects_present": [k for k, v in defects.items() if v["present"]],
        "total_defect_area_pct": round(sum(v["area_pct"] for v in defects.values()), 2),
        "sugar_spots": summary["sugar_spot"],
    }, masks


def skin_condition(rgb, mask, sugar_spots=None):
    t = texture_features(rgb, mask)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    fruit = mask > 0
    V, S = hsv[..., 2][fruit], hsv[..., 1][fruit]
    # Gloss: bright, desaturated highlights relative to the fruit's own brightness
    # (an absolute V>225 test misses highlights on dark dates).
    thr = max(np.median(V) + 3.0 * (np.percentile(V, 75) - np.percentile(V, 25)), np.percentile(V, 95))
    highlight = (V >= thr) & (S < np.median(S))
    gloss = float(highlight.mean())
    contrast = float((V[highlight].mean() - np.median(V)) / 255) if highlight.any() else 0.0
    gloss_index = gloss * contrast * 100
    level = lambda v, lo, hi: "low" if v < lo else "medium" if v < hi else "high"
    out = {
        "wrinkle_index": t["wrinkle_index"],
        "wrinkling": t["wrinkling"],
        "gloss_index": round(gloss_index, 3),
        "gloss": level(gloss_index, 0.3, 1.0),
        "surface": t["surface"],
        "glcm_homogeneity": t["glcm_homogeneity"],
    }
    if sugar_spots is not None:
        out["sugar_spots"] = {**sugar_spots, "level": "none" if not sugar_spots["present"]
                              else level(sugar_spots["area_pct"], 1.0, 3.0)}
    return out


def maturity_by_colour(rgb, mask):
    """Rule-based ripeness from skin colour, for what colour can tell: Immature
    (Kimri) dates are green, Khalal fully yellow or red, and Rutab ripen from the
    tip so part of the skin is still yellow/red. A fully brown fruit is Rutab or
    Tamar; colour does not separate those (on Alhamdan Rutab vs Tamar photos a
    brown/wrinkle rule scored 38%), so the maturity CNN makes that call."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV_FULL)[mask > 0].astype(float)
    h, s, v = hsv[:, 0] * 360 / 255, hsv[:, 1] / 255, hsv[:, 2] / 255
    green = ((h >= 65) & (h < 160) & (s > 0.25) & (v > 0.2)).mean()
    khalal = ((((h >= 35) & (h < 65)) | (h < 12) | (h >= 340)) & (s > 0.45) & (v > 0.5)).mean()
    shares = {"green": round(float(green), 3), "khalal_yellow_red": round(float(khalal), 3)}
    if green > 0.5:
        stage = "Immature"
    elif khalal > 0.6:
        stage = "Khalal"
    elif khalal > 0.15:
        stage = "Rutab (partly ripened)"
    else:
        stage = "Rutab or Tamar (fully ripened)"
    return {"stage": stage, "skin_shares": shares}


def analyse_fruit(rgb, mask, mm_per_px=None, variety=None, px_scale=1.0, closeup=None):
    """rgb/mask: overview image (geometry). closeup: (rgb, mask) of the fruit at higher
    resolution for colour, defects and skin; defaults to the overview."""
    geom = geometry(mask, mm_per_px, px_scale)
    crgb, cmask = closeup if closeup is not None else (rgb, mask)
    defects, defect_masks = surface_defects(crgb, cmask)
    skin = skin_condition(crgb, cmask, defects.get("sugar_spots"))
    return {
        "geometry": geom,
        "size": size_class(geom, variety),
        "estimated_weight_g": estimate_weight(geom),
        "color_uniformity": color_uniformity(crgb, cmask),
        "surface_defects": defects,
        "skin": skin,
        "maturity_by_colour": maturity_by_colour(crgb, cmask),
    }, defect_masks


# ---------------------------------------------------------------- checks / calibration
def _grade_row(path):
    rgb = load_rgb(path)
    rgb = cv2.copyMakeBorder(rgb, 30, 30, 30, 30, cv2.BORDER_REPLICATE)
    mask = segment_fruit(rgb)
    d, _ = surface_defects(rgb, mask)
    if "error" in d:
        return None
    s = skin_condition(rgb, mask)
    row = {f"{k}_pct": v["area_pct"] for k, v in d["defects"].items()}
    row.update(total_defect_pct=d["total_defect_area_pct"], sugar_spot_pct=d["sugar_spots"]["area_pct"],
               wrinkle_index=s["wrinkle_index"], gloss_index=s["gloss_index"],
               color_uniformity=color_uniformity(rgb, mask)["uniformity_score"])
    return row


def check_defects():
    """No defect labels exist, so check the rules against the grade labels: a Grade-3
    fruit should show more defect area than a Grade-1 fruit of the same variety."""
    import concurrent.futures as cf

    df = pd.read_csv(config.QUALITY_DIR / "grading_index.csv")
    df = df[df.variety.isin(["Gajar", "Kupro"])]  # the two varieties with all three grades
    with cf.ProcessPoolExecutor() as pool:
        rows = list(pool.map(_grade_row, df.path, chunksize=8))
    feats = pd.DataFrame([r or {} for r in rows], index=df.index)
    df = pd.concat([df, feats], axis=1).dropna(subset=["total_defect_pct"])
    out = config.OUTPUT_DIR / "quality"
    out.mkdir(parents=True, exist_ok=True)
    df.assign(path=df.path.map(lambda p: os.path.relpath(p, config.ROOT))).to_csv(
        out / "defect_features_by_grade.csv", index=False)
    cols = [c for c in feats.columns]
    table = df.groupby(["variety", "grade"])[cols].mean().round(3)
    from scipy.stats import spearmanr

    rho = {c: {v: round(float(spearmanr(g.grade.str[-1].astype(int), g[c])[0]), 3)
               for v, g in df.groupby("variety")} for c in cols}
    (out / "defect_check.json").write_text(json.dumps(
        {"mean_by_grade": json.loads(table.to_json(orient="index")), "spearman_with_grade": rho}, indent=2))
    print(table.T.to_string(), "\n\nSpearman(feature, grade number):\n", pd.DataFrame(rho).to_string())


def calibrate_weight(csv_path):
    """Fit WEIGHT_K from fruits weighed on a scale: image,weight_g[,mm_per_px]."""
    df = pd.read_csv(csv_path)
    lw2 = []
    for r in df.itertuples():
        rgb = load_rgb(r.image)
        mmpp, corners = find_scale(rgb)
        if getattr(r, "mm_per_px", None):  # given per original pixel; load_rgb resized the image
            mmpp = r.mm_per_px * max(Image.open(r.image).size) / max(rgb.shape[:2])
        if not mmpp:
            raise SystemExit(f"{r.image}: no ArUco marker found and no mm_per_px given")
        g = geometry(fruit_instances(hide_marker(rgb, corners))[0], mmpp)
        lw2.append(g["length_mm"] * g["width_mm"] ** 2)
    lw2 = np.array(lw2)
    k = float((lw2 * df.weight_g).sum() / (lw2**2).sum())  # least squares through the origin
    err = np.abs(k * lw2 - df.weight_g) / df.weight_g
    print(json.dumps({"WEIGHT_K": k, "n": len(df), "mean_abs_pct_error": round(100 * float(err.mean()), 1)}, indent=2))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("image", nargs="?")
    ap.add_argument("--mm-per-px", type=float, help="mm per pixel of the original image")
    ap.add_argument("--variety")
    ap.add_argument("--fit-size", action="store_true")
    ap.add_argument("--check-defects", action="store_true")
    ap.add_argument("--calibrate-weight")
    a = ap.parse_args()
    if a.fit_size:
        fit_size()
    elif a.check_defects:
        check_defects()
    elif a.calibrate_weight:
        calibrate_weight(a.calibrate_weight)
    elif a.image:
        rgb = load_rgb(a.image)
        px_scale = max(rgb.shape[:2]) / max(Image.open(a.image).size)
        mmpp, corners = find_scale(rgb)
        if a.mm_per_px:
            mmpp = a.mm_per_px / px_scale
        rgb = hide_marker(rgb, corners)
        masks = fruit_instances(rgb)
        print(json.dumps(analyse_fruit(rgb, masks[0], mmpp, a.variety, px_scale)[0], indent=2))
    else:
        ap.print_help(sys.stderr)

In [ ]:
%%writefile src/train.py
"""Two-stage transfer learning: (1) frozen backbone, train head; (2) fine-tune
the backbone (all or the top N layers) with a cosine learning-rate schedule.
The best checkpoint by validation loss across both stages is kept.

Usage:
  python src/train.py --backbone mobilenetv3
  python src/train.py --data grade --run grade_mnv3      # Phase 2 quality grade
  python src/train.py --backbone efficientnetv2s --img-size 384 --ft-layers 0 --ft-lr 1e-4 --run effv2s_384
"""
import argparse
import json

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

import config
from data import get_datasets
from model import BACKBONES, build_model, unfreeze_top


def plot_history(hist, split_epoch, path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, key in zip(axes, ["loss", "accuracy"]):
        ax.plot(hist[key], label=f"train {key}")
        ax.plot(hist[f"val_{key}"], label=f"val {key}")
        ax.axvline(split_epoch - 0.5, color="gray", ls="--", label="start fine-tune")
        ax.set_xlabel("epoch")
        ax.set_title(key)
        ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=120)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="variety", choices=sorted(config.DATASETS))
    ap.add_argument("--backbone", default=config.BACKBONE, choices=sorted(BACKBONES))
    ap.add_argument("--img-size", type=int, help="default: the backbone's native size")
    ap.add_argument("--run", help="artifact name (default: backbone name)")
    ap.add_argument("--batch-size", type=int, default=config.BATCH_SIZE)
    ap.add_argument("--head-epochs", type=int, default=config.HEAD_EPOCHS)
    ap.add_argument("--ft-epochs", type=int, default=config.FINETUNE_EPOCHS)
    ap.add_argument("--ft-layers", type=int, default=config.FINETUNE_LAST_N_LAYERS, help="0 = whole backbone")
    ap.add_argument("--ft-lr", type=float, default=config.FINETUNE_LR)
    ap.add_argument("--label-smoothing", type=float, default=0.0)
    ap.add_argument("--mixed-precision", action="store_true", help="float16 compute (GPU only)")
    args = ap.parse_args()
    img_size = args.img_size or BACKBONES[args.backbone][1]
    run = args.run or (args.backbone if args.data == "variety" else f"{args.data}_{args.backbone}")

    if args.mixed_precision:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
    tf.keras.utils.set_random_seed(config.SEED)
    config.MODEL_DIR.mkdir(parents=True, exist_ok=True)
    train_ds, val_ds, _, class_names = get_datasets(img_size, args.batch_size, one_hot=True,
                                                        data_dir=config.DATASETS[args.data])
    labels = np.concatenate([y.numpy().argmax(1) for _, y in train_ds])
    weights = compute_class_weight("balanced", classes=np.arange(len(class_names)), y=labels)
    class_weight = dict(enumerate(weights))
    print(f"run={run} backbone={args.backbone} img={img_size} classes={class_names}")

    model = build_model(len(class_names), args.backbone, img_size)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=args.label_smoothing)
    ckpt = config.MODEL_DIR / f"{run}_best.keras"

    print("\n=== Stage 1: frozen backbone ===")
    model.compile(tf.keras.optimizers.Adam(config.HEAD_LR), loss, metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=args.head_epochs, class_weight=class_weight,
                   callbacks=[
                       tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True),
                       tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
                   ])
    best_stage1 = min(h1.history["val_loss"])

    what = "whole backbone" if args.ft_layers <= 0 else f"last {args.ft_layers} backbone layers"
    print(f"\n=== Stage 2: fine-tune {what} ===")
    unfreeze_top(model, args.ft_layers)
    steps = args.ft_epochs * len(train_ds)
    schedule = tf.keras.optimizers.schedules.CosineDecay(
        0.0, steps, warmup_target=args.ft_lr, warmup_steps=max(1, int(0.1 * steps)), alpha=0.01)
    model.compile(tf.keras.optimizers.AdamW(schedule, weight_decay=1e-4), loss, metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=args.ft_epochs, class_weight=class_weight,
                   callbacks=[
                       # only overwrite the stage-1 checkpoint if fine-tuning actually beats it
                       tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True,
                                                          initial_value_threshold=best_stage1),
                       tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10),
                   ])

    best = tf.keras.models.load_model(ckpt)
    val_loss, val_acc = best.evaluate(val_ds, verbose=0)
    hist = {k: h1.history[k] + h2.history[k] for k in ("loss", "accuracy", "val_loss", "val_accuracy")}
    meta = {"run": run, "data": args.data, "backbone": args.backbone, "img_size": img_size, "class_names": class_names,
            "args": vars(args), "best_val_loss": val_loss, "best_val_accuracy": val_acc,
            "stage1_epochs": len(h1.history["loss"]), "history": hist}
    (config.MODEL_DIR / f"{run}_meta.json").write_text(json.dumps(meta, indent=2))
    (config.OUTPUT_DIR / "training_curves").mkdir(parents=True, exist_ok=True)
    plot_history(hist, len(h1.history["loss"]), config.OUTPUT_DIR / "training_curves" / f"{run}_training_curves.png")
    print(f"saved {ckpt}  best val_acc={val_acc:.4f} val_loss={val_loss:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/download_zenodo.py
"""Download the Date Fruit Dataset (Zenodo record 4639543) into data/raw/.

Resumable (complete files are skipped). Zenodo rate-limits parallel downloads,
so each file is size-checked, failures are retried with backoff, and the script
exits non-zero unless every file is on disk at the end.
"""
import concurrent.futures as cf
import json
import pathlib
import socket
import sys
import time
import urllib.request

RECORD = "https://zenodo.org/api/records/4639543/files"
OUT = pathlib.Path(__file__).resolve().parents[1] / "data" / "raw"


def complete(entry):
    dest = OUT / entry["key"]
    return dest.exists() and dest.stat().st_size == entry["size"]


def fetch(entry):
    if complete(entry):
        return entry["key"], "skip"
    dest = OUT / entry["key"]
    tmp = dest.with_name(dest.name + ".part")
    err = None
    for attempt in range(8):
        try:
            urllib.request.urlretrieve(entry["links"]["content"], tmp)
            if tmp.stat().st_size != entry["size"]:
                raise OSError(f"got {tmp.stat().st_size} bytes, expected {entry['size']}")
            tmp.rename(dest)
            return entry["key"], "ok"
        except Exception as e:  # noqa: BLE001 - network errors of all kinds are retried
            err = e
            headers = getattr(e, "headers", None)
            wait = headers.get("Retry-After") if headers else None
            time.sleep(float(wait) if wait else min(60, 3 * 2**attempt))
    tmp.unlink(missing_ok=True)
    return entry["key"], f"FAILED: {err}"


def main():
    socket.setdefaulttimeout(60)
    OUT.mkdir(parents=True, exist_ok=True)
    entries = json.load(urllib.request.urlopen(RECORD))["entries"]
    with cf.ThreadPoolExecutor(3) as pool:
        for key, status in pool.map(fetch, entries):
            if status.startswith("FAILED"):
                print(key, status, flush=True)
    missing = [e["key"] for e in entries if not complete(e)]
    print(f"{len(entries) - len(missing)}/{len(entries)} files complete")
    sys.exit(1 if missing else 0)


if __name__ == "__main__":
    main()

## Data: Zenodo 4639543 + Kaggle *Date Fruit Image Dataset in Controlled Environment*

In [ ]:
if not os.path.exists("data/kaggle/alhamdan/Ajwa"):
    !mkdir -p data/kaggle data/raw
    !curl -sL -o data/kaggle/alhamdan_dates.zip https://www.kaggle.com/api/v1/datasets/download/wadhasnalhamdan/date-fruit-image-dataset-in-controlled-environment
    assert os.path.getsize("data/kaggle/alhamdan_dates.zip") > 1e9, "Kaggle download failed: download the dataset zip manually and upload it to data/kaggle/alhamdan_dates.zip"
    !unzip -q -o data/kaggle/alhamdan_dates.zip -d data/kaggle/alhamdan && rm data/kaggle/alhamdan_dates.zip
# Zenodo rate-limits downloads; the script resumes, so retry a few passes until all 478 files are there.
for attempt in range(4):
    if sh("python scripts/download_zenodo.py", grep=["complete", "FAILED"]) == 0:
        break
    time.sleep(30)
else:
    raise RuntimeError("Zenodo download incomplete - re-run this cell")
assert len(os.listdir("data/kaggle/alhamdan")) >= 9, "Kaggle data missing"

In [ ]:
sh("python src/prepare_data.py", check=True)

## Backbone sweep
Every run: frozen-backbone head training, then **whole-backbone** fine-tuning with cosine LR,
label smoothing 0.1, mixed precision. Best checkpoint by validation loss.

In [ ]:
RUNS = [
    # run name,          backbone,           img, fine-tune LR
    ("mnv3_224",         "mobilenetv3",      224, 1e-4),
    ("effv2b0_260",      "efficientnetv2b0", 260, 1e-4),
    ("effv2s_384",       "efficientnetv2s",  384, 5e-5),
    ("convnext_t_288",   "convnext_tiny",    288, 5e-5),
    ("convnext_s_288",   "convnext_small",   288, 3e-5),
]
os.makedirs("outputs/logs", exist_ok=True)
for run, bb, img, lr in RUNS:
    if os.path.exists(f"outputs/models/{run}_meta.json"):
        print("skip (done):", run); continue
    t = time.time()
    log = f"outputs/logs/train_{run}.log"
    sh(f"python src/train.py --backbone {bb} --img-size {img} --run {run} --ft-layers 0 --ft-lr {lr} "
       f"--label-smoothing 0.1 --head-epochs 15 --ft-epochs 40 --mixed-precision > {log} 2>&1")
    print(f"{run}: {(time.time()-t)/60:.1f} min")
    sh(f"tail -1 {log}", grep=["saved", "Error"])  # full log: outputs/logs/train_<run>.log
assert any(os.path.exists(f"outputs/models/{r}_meta.json") for r, *_ in RUNS), "every training run failed - see outputs/logs/"

## Model selection on the validation split (single models and ensembles, with flip-TTA)

In [ ]:
done = [r for r, *_ in RUNS if os.path.exists(f"outputs/models/{r}_meta.json")]
def val_score(runs):
    name = "+".join(runs) + "_tta"
    path = f"outputs/eval/{name}_val_metrics.json"
    if not os.path.exists(path):
        sh(f"python src/evaluate.py --run {' '.join(runs)} --tta --split val", grep=["Error"])
    m = json.load(open(path))
    return (m["accuracy"], m["f1_macro"], -m["loss"]), m
scores = {(r,): val_score([r]) for r in done}
top = sorted(scores, key=lambda k: scores[k][0], reverse=True)[:3]
for k in (2, 3):
    for combo in itertools.combinations([t[0] for t in top], k):
        scores[combo] = val_score(list(combo))
print(f"{'candidate':55s} val_acc  val_f1  val_loss")
for k in sorted(scores, key=lambda k: scores[k][0], reverse=True):
    m = scores[k][1]
    print(f"{'+'.join(k):55s} {m['accuracy']:.4f}  {m['f1_macro']:.4f}  {m['loss']:.4f}")
best = max(scores, key=lambda k: (scores[k][0], -len(k)))  # tie -> fewer models
best_single = max((k for k in scores if len(k) == 1), key=lambda k: scores[k][0])
print("\nSELECTED:", "+".join(best), "| best single:", best_single[0])
json.dump({"selected": list(best), "best_single": best_single[0]}, open("outputs/selection.json", "w"))

## Final test-set scores (touched once)

In [ ]:
for runs in dict.fromkeys(["mnv3_224", best_single[0], "+".join(best)]):
    sh(f"python src/evaluate.py --run {runs.replace('+', ' ')} --tta --split test", grep=["] {", "by source", "Error"])

## Explainability + feature profiles + example prediction for the best single model

In [ ]:
run = best_single[0]
sh(f"python src/explain.py --run {run} --per-class 1 > outputs/logs/explain.log 2>&1; tail -14 outputs/logs/explain.log")
sh("python src/features.py --class-profiles > outputs/logs/features.log 2>&1; tail -3 outputs/logs/features.log")
ex = sorted(glob.glob("data/processed/test/Al-ajwa/*.jpg"))[0]
sh(f'python src/predict.py "{ex}" --run {run} 2>&1 | grep -A12 predicted_variety')

In [ ]:
!cd /content/date-fruit-quality-ml && zip -qr /content/results.zip outputs data/splits.csv
!ls -lh /content/results.zip
try:  # optional copy to Drive if it was mounted (Colab extension: "Mount Google Drive")
    if os.path.isdir("/content/drive/MyDrive"):
        !cp /content/results.zip /content/drive/MyDrive/date_fruit_results.zip && echo "copied to Drive"
except Exception as e:
    print(e)